In [1]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Project paths
PROJECT_ROOT  = r'C:\Users\chiku\OneDrive\Documents\ecom_dynamic_pricing_optimization'
DATA_DIR      = os.path.join(PROJECT_ROOT, 'data')
PROCESSED_DIR = os.path.join(PROJECT_ROOT, 'data', 'processed')
SRC_DIR       = os.path.join(PROJECT_ROOT, 'src')

os.makedirs(PROCESSED_DIR, exist_ok=True)
sys.path.insert(0, SRC_DIR)  # Makes src/ importable without pip install -e .

print('Setup complete ✅')
print(f'Python path includes: {SRC_DIR}')

Setup complete ✅
Python path includes: C:\Users\chiku\OneDrive\Documents\ecom_dynamic_pricing_optimization\src


In [2]:
# Load pre-aggregated data — no need to re-merge 32M rows every time
product_demand = pd.read_parquet(os.path.join(PROCESSED_DIR, 'product_demand.parquet'))
transactions   = pd.read_parquet(os.path.join(PROCESSED_DIR, 'transactions_sample.parquet'))

print(f'product_demand shape:  {product_demand.shape}')
print(f'transactions shape:    {transactions.shape}')
print(f'\nproduct_demand columns: {list(product_demand.columns)}')
print(f'transactions columns:   {list(transactions.columns)}')

product_demand shape:  (49677, 6)
transactions shape:    (500000, 9)

product_demand columns: ['product_id', 'department', 'units_sold', 'avg_price', 'price_std', 'n_orders']
transactions columns:   ['product_id', 'department', 'aisle', 'price', 'base_price', 'order_dow', 'order_hour_of_day', 'days_since_prior_order', 'add_to_cart_order']


In [4]:
# ─── Feature Engineering ──────────────────────────────────────────────────────

# 1. LOG TRANSFORMATIONS
# Economics: log-log specification makes slope = elasticity directly
transactions['log_price'] = np.log(transactions['price'])
transactions['log_units'] = np.log1p(transactions['add_to_cart_order'])

# 2. PRICE RELATIVE TO CATEGORY MEDIAN
# Economics: controls for quality differences across departments
#            A $5 item is cheap in meat but expensive in spices
category_median_price = transactions.groupby('department')['price'].transform('median')
transactions['price_vs_category'] = transactions['price'] / category_median_price
transactions['log_price_vs_category'] = np.log(transactions['price_vs_category'])

# 3. TEMPORAL FEATURES (demand shifters — move the curve, not along it)
transactions['is_weekend'] = transactions['order_dow'].isin([0, 6]).astype(int)
transactions['is_morning']  = transactions['order_hour_of_day'].between(6, 11).astype(int)
transactions['is_afternoon'] = transactions['order_hour_of_day'].between(12, 17).astype(int)
transactions['is_evening']  = transactions['order_hour_of_day'].between(18, 22).astype(int)

# 4. PAYDAY PROXIMITY (demand shifter — income effect)
# Economics: demand for normal goods rises around payday (1st and 15th)
# days_since_prior_order as a proxy — short gaps suggest recent payday
transactions['is_frequent_shopper'] = (transactions['days_since_prior_order'] < 7).astype(int)

# 5. PRODUCT POPULARITY (controls for quality/brand — key confounder)
# Economics: popular products command price premiums (brand equity)
#            Must control for this or OLS conflates quality with price effect
popularity = product_demand.set_index('product_id')['units_sold']
transactions['log_popularity'] = np.log1p(
    transactions['product_id'].map(popularity).fillna(0)
)

print('Features created ✅')
print(f'\nNew columns added: {[c for c in transactions.columns if c not in ["product_id", "department", "aisle", "price", "base_price", "order_dow", "order_hour_of_day", "days_since_prior_order", "add_to_cart_order"]]}')
print(f'\nFinal transactions shape: {transactions.shape}')
print(f'\nSample row:')
print(transactions[['log_price', 'price_vs_category', 'is_weekend', 'log_popularity']].head(3).to_string())

Features created ✅

New columns added: ['log_price', 'log_units', 'price_vs_category', 'log_price_vs_category', 'is_weekend', 'is_morning', 'is_afternoon', 'is_evening', 'is_frequent_shopper', 'log_popularity']

Final transactions shape: (500000, 19)

Sample row:
   log_price  price_vs_category  is_weekend  log_popularity
0   1.526056           1.385542           1        6.559615
1   0.955511           0.841424           0       11.071159
2   0.425268           0.495146           0       11.286903


In [ ]:
# ─── Define DML matrices ──────────────────────────────────────────────────────

# Y: Outcome — what we're trying to explain
Y_col = 'log_units'

# T: Treatment — the causal variable we want the effect of
T_col = 'log_price'

# W: Controls — confounders to partial out (the FWL "nuisance" variables)
# These explain BOTH price and demand, creating the endogeneity we must remove
W_cols = [
    'price_vs_category',       # Quality/positioning confounder
    'log_popularity',          # Brand equity confounder
    'is_weekend',              # Temporal demand shifter
    'is_morning',              # Hour-of-day demand shifter
    'is_afternoon',
    'is_evening',
    'is_frequent_shopper',     # Income/loyalty proxy
    'days_since_prior_order',  # Purchase cycle position
]

# X: Heterogeneity features — contexts where elasticity might differ
# These go into the CATE model to estimate conditional elasticities
X_cols = [
    'price_vs_category',   # Elasticity differs for premium vs budget products
    'is_weekend',          # Weekend shoppers may be less price sensitive
    'log_popularity',      # Popular products have more inelastic demand
]

# Encode department as numeric for the model
transactions['department_code'] = pd.Categorical(transactions['department']).codes
W_cols.append('department_code')

print('DML matrix definitions:')
print(f'  Y  (outcome):      {Y_col}')
print(f'  T  (treatment):    {T_col}')
print(f'  W  (controls):     {len(W_cols)} features → {W_cols}')
print(f'  X  (heterog.):     {len(X_cols)} features → {X_cols}')

# ─── Save processed data ──────────────────────────────────────────────────────
transactions.to_parquet(os.path.join(PROCESSED_DIR, 'train.parquet'), index=False)

# Save column definitions for use in notebooks 03 and 04
import json
col_definitions = {
    'Y_col': Y_col,
    'T_col': T_col,
    'W_cols': W_cols,
    'X_cols': X_cols,
}
with open(os.path.join(PROCESSED_DIR, 'col_definitions.json'), 'w') as f:
    json.dump(col_definitions, f, indent=2)

print(f'\n✅ Saved train.parquet       — {len(transactions):,} rows')
print(f'✅ Saved col_definitions.json — column names for DML')
print(f'\nNotebook 02 complete — ready for notebook 03 (baseline models)')

DML matrix definitions:
  Y  (outcome):      log_units
  T  (treatment):    log_price
  W  (controls):     9 features → ['price_vs_category', 'log_popularity', 'is_weekend', 'is_morning', 'is_afternoon', 'is_evening', 'is_frequent_shopper', 'days_since_prior_order', 'department_code']
  X  (heterog.):     3 features → ['price_vs_category', 'is_weekend', 'log_popularity']

✅ Saved train.parquet       — 500,000 rows
✅ Saved col_definitions.json — column names for DML

Notebook 02 complete — ready for notebook 03 (baseline models)


: 